In [40]:
from langchain_core.tools import tool
from dotenv import load_dotenv
import requests
from langchain_core.messages import HumanMessage
load_dotenv()
import os
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

In [ ]:
@tool
def get_conversion_factor(base_currency:str,target_currency:str)->float:
    """This is fetches the currency conversion factor between base_currency and target_currency"""
    url = f" https://v6.exchangerate-api.com/v6/9df6e706193bf1f41ce9c91e/pair/{base_currency}/{target_currency}"
    response = requests.get(url)
    return response.json()


@tool
def convert(amount:int,conversion_factor:float)->float:
    """Given a currency conversion rate this function calculates the target currency value from a given base currency value."""

    return amount * conversion_factor


In [42]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1775779202,
 'time_last_update_utc': 'Fri, 10 Apr 2026 00:00:02 +0000',
 'time_next_update_unix': 1775865602,
 'time_next_update_utc': 'Sat, 11 Apr 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 92.812}

In [ ]:
convert.invoke({'amount':10,'conversion_factor':92.5746})

925.7460000000001

In [44]:
llm = ChatGroq(model='llama-3.3-70b-versatile')
# llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')

In [45]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [46]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [47]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [48]:

ai_message = llm_with_tools.invoke(messages)
ai_message

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=get_conversion_factor{"base_currency": "INR", "target_currency": "USD"}</function>\n'}}

In [ ]:
messages.append(ai_message)


In [ ]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'wqb8hkjss',
  'type': 'tool_call'}]

In [49]:
# import json
# for tool_call in ai_message.tool_calls:
#     if tool_call['name'] == 'get_conversion_factor':
#         tool_messgae1 = get_conversion_factor.invoke(tool_call)
#         conversion_factor = json.loads(tool_messgae1.content)['conversion_rate']
#         messages.append(tool_messgae1)

#     if tool_call['name'] == 'convert':
#         tool_call['args']['conversion_factor'] = conversion_factor
#         tool_messgae2 = convert.invoke(tool_call)
#         messages.append(tool_messgae2)

tools_map = {
    'get_conversion_factor': get_conversion_factor,
    'convert': convert,
}

conversion_factor = None  # track across turns

while True:
    ai_message = llm_with_tools.invoke(messages)
    messages.append(ai_message)

    # If no tool calls remain, the LLM has produced a final text response
    if not ai_message.tool_calls:
        break

    for tool_call in ai_message.tool_calls:
        tool_name = tool_call['name']

        if tool_name == 'get_conversion_factor':
            result = get_conversion_factor.invoke(tool_call)
            conversion_factor = json.loads(result.content)['conversion_rate']
            messages.append(result)

        elif tool_name == 'convert':
            # Inject conversion_factor if the LLM didn't include it
            if conversion_factor is not None and 'conversion_factor' not in tool_call['args']:
                tool_call['args']['conversion_factor'] = conversion_factor
            result = convert.invoke(tool_call)
            messages.append(result)

print("\n=== Final LLM Response ===")
print(ai_message.content)


=== Final LLM Response ===
The conversion factor between INR and USD is 0.013. Based on this conversion factor, 10 INR is equivalent to 0.13 USD.


In [ ]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'wqb8hkjss', 'function': {'arguments': '{"base_currency":"INR","target_currency":"USD"}', 'name': 'get_conversion_factor'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 352, 'total_tokens': 398, 'completion_time': 0.141618468, 'completion_tokens_details': None, 'prompt_time': 0.018262611, 'prompt_tokens_details': None, 'queue_time': 0.161841459, 'total_time': 0.159881079}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d75f4-439f-78b2-ba20-7927c1259f35-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 

In [ ]:
final_response = llm_with_tools.invoke(messages)
print(final_response.content)